# Reproduce all experiments (Colab, T4 GPU)
Runtime → Change runtime type → **T4 GPU**, then Runtime → **Run all**. Roughly 20–40 minutes.
At the end it updates `README.md` with the results table, saves figures to `results/figures/`, and downloads a zip you can commit.

In [ ]:
REPO_URL = "https://github.com/KingQuinn/transformer-from-scratch.git"  # <- change if your repo name differs
!git clone -q $REPO_URL repo
%cd repo
!nvidia-smi --query-gpu=name --format=csv,noheader
!pip -q install -r requirements.txt

## 1. Unit tests (causality, padding invariance, LayerNorm, PE, parameter count, data splits)

In [ ]:
!python -m pytest -q

## 2. Smoke test (1 epoch, tiny data), so crashes show up in seconds rather than after 20 minutes

In [ ]:
!python -m transformer.train --task addition --epochs 1 --n_train 512 --n_val 64 --n_test 64 --n_ood 64 --out_dir /tmp/smoke

## 3. Main experiments: 3 tasks × 3 seeds

In [ ]:
import subprocess, sys
RUNS = {
    "copy":     "--epochs 30 --patience 5",
    "reverse":  "--epochs 30 --patience 5",
    "addition": "--epochs 80 --patience 10 --n_train 50000",
}
for task, extra in RUNS.items():
    for seed in (0, 1, 2):
        cmd = f"{sys.executable} -m transformer.train --task {task} --seed {seed} {extra}"
        print("\n>>>", cmd, flush=True)
        r = subprocess.run(cmd, shell=True)
        if r.returncode != 0:
            raise RuntimeError(f"{task} seed {seed} failed")
# If Colab disconnects: re-run this cell after adding --resume to `extra`; runs continue from checkpoints/<run>/last.pt

## 4. Figures, tables, and README update

In [ ]:
!python scripts/analyze.py --tasks copy reverse addition --seeds 0 1 2

In [ ]:
from IPython.display import Image, display, Markdown
display(Markdown(open("results/RESULTS.md").read()))
for f in ["training_curves", "accuracy_by_length", "cross_attention", "positional_encoding", "residual_gradients"]:
    print(f); display(Image(f"results/figures/{f}.png"))

## 5. Download everything to commit (README, results, figures, best checkpoints)

In [ ]:
!zip -qr /content/results_to_commit.zip README.md results checkpoints/*/best.pt
from google.colab import files
files.download("/content/results_to_commit.zip")